In [ ]:
# ==============================================================================
# PARAMETERS
# ==============================================================================

SAMPLE_INDEX = 1
SAMPLE_ID = "sample"
SAMPLE_DIR_NAME = "001_sample"

INPUT_PATH = "scimap_gated.h5ad"
OUTPUT_DIR = "contingency_output"

# Only scientific plotting parameter
TOP_N_POPULATIONS = 30

PDF_DPI = 300
HEATMAP_CMAP = "Blues"

In [ ]:
# ==============================================================================
# IMPORTS AND LOAD FULL ANNDATA
# ==============================================================================

from pathlib import Path
import re
import gc
import h5py
import numpy as np
import pandas as pd
import anndata as ad

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

INPUT_PATH = str(INPUT_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)

PLOT_DIR = OUTPUT_DIR / "plots" / "contingency"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


# Compatibility patch on the temporary local copy made by Nextflow.
# This does not touch the upstream SCIMAP output.
with h5py.File(INPUT_PATH, "a") as f:
    if "uns/log1p" in f:
        del f["uns/log1p"]
        print("Patched /uns/log1p")

adata = ad.read_h5ad(INPUT_PATH)

print("=" * 80)
print("Loaded full AnnData")
print("=" * 80)
print("Input:", INPUT_PATH)
print("AnnData:", adata)
print("obs columns:", len(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))
print("layers:", list(adata.layers.keys()))
print("=" * 80)

In [ ]:
# ==============================================================================
# DETECT TOP SCIMAP POPULATIONS AND LEIDEN CLUSTERS
# ==============================================================================

population_key = "scimap_population"

if population_key not in adata.obs.columns:
    raise KeyError(
        f"Missing {population_key!r} in adata.obs. Available columns:\n"
        + "\n".join(map(str, adata.obs.columns))
    )

leiden_keys = [
    c for c in adata.obs.columns
    if re.match(r"^leiden_.*_res_.*$", str(c))
]

if len(leiden_keys) == 0:
    raise KeyError(
        "No Leiden columns found. Expected columns like leiden_arcsinh_res_0_5."
    )


def leiden_sort_key(key):
    key = str(key)
    norm_order = {"arcsinh": 0, "log1p": 1}
    norm = "other"
    res = -1.0

    m_norm = re.search(r"^leiden_([^_]+)_res_", key)
    if m_norm:
        norm = m_norm.group(1)

    m_res = re.search(r"_res_([0-9_\.]+)$", key)
    if m_res:
        try:
            res = float(m_res.group(1).replace("_", "."))
        except Exception:
            res = -1.0

    return (norm_order.get(norm, 99), res, key)


leiden_keys = sorted(leiden_keys, key=leiden_sort_key)

top_n = int(TOP_N_POPULATIONS)

adata.obs["_contingency_population"] = (
    adata.obs[population_key]
    .astype("string")
    .fillna("missing_population")
    .astype(str)
)

population_counts = adata.obs["_contingency_population"].value_counts()

top_populations = population_counts.head(top_n).index.tolist()

obs_plot = adata.obs[
    adata.obs["_contingency_population"].isin(top_populations)
].copy()

print("=" * 80)
print("Contingency inputs")
print("=" * 80)
print("Population key:", population_key)
print("TOP_N_POPULATIONS:", top_n)
print("Cells kept for top populations:", obs_plot.shape[0], "/", adata.n_obs)
print()
print("Top populations:")
print(population_counts.head(top_n).to_string())
print()
print("Leiden keys:")
for k in leiden_keys:
    print(" -", k, "| all clusters:", adata.obs[k].astype(str).nunique())
print("=" * 80)

In [ ]:
# ==============================================================================
# HELPERS
# ==============================================================================

def cluster_sort_key(x):
    s = str(x)
    try:
        return (0, int(s))
    except Exception:
        try:
            return (0, float(s))
        except Exception:
            return (1, s)


def safe_filename(x):
    s = str(x)
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def pretty_leiden_title(leiden_key):
    key = str(leiden_key)
    norm = "unknown"
    res = "unknown"

    m_norm = re.search(r"leiden_([^_]+)_res_", key)
    if m_norm:
        norm = m_norm.group(1)

    m_res = re.search(r"_res_([0-9_\.]+)$", key)
    if m_res:
        res = m_res.group(1).replace("_", ".")

    nclus = adata.obs[leiden_key].astype(str).nunique()

    return f"{norm} normalization | Leiden resolution {res} | {nclus} clusters"


def make_counts(leiden_key):
    tmp = obs_plot[["_contingency_population", leiden_key]].copy()
    tmp[leiden_key] = tmp[leiden_key].astype(str)

    counts = pd.crosstab(
        tmp["_contingency_population"],
        tmp[leiden_key],
        dropna=False,
    )

    # Keep top populations in abundance order
    counts = counts.reindex(index=top_populations, fill_value=0)

    # Keep all clusters from the full AnnData, even if a top population has zero cells there
    all_clusters = sorted(
        adata.obs[leiden_key].astype(str).dropna().unique().tolist(),
        key=cluster_sort_key,
    )

    counts = counts.reindex(columns=all_clusters, fill_value=0)

    counts.columns = [f"C{c}" for c in counts.columns]

    return counts.astype(int)


def plot_counts_heatmap(counts, leiden_key, out_pdf):
    arr = counts.to_numpy(dtype=float)

    n_rows, n_cols = arr.shape

    fig_w = max(9, min(28, 5 + n_cols * 0.45))
    fig_h = max(7, min(20, 3 + n_rows * 0.42))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        arr,
        cmap=str(HEATMAP_CMAP),
        aspect="auto",
        interpolation="nearest",
    )

    ax.set_title(
        "Raw contingency counts: top SCIMAP populations × Leiden clusters\n"
        + pretty_leiden_title(leiden_key),
        fontsize=11,
        pad=12,
    )

    ax.set_xlabel("Leiden cluster", fontsize=9)
    ax.set_ylabel(f"Top {top_n} SCIMAP gated populations", fontsize=9)

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(counts.columns.astype(str), rotation=45, ha="right", fontsize=7)

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(counts.index.astype(str), fontsize=7)

    # Write raw counts inside cells
    max_val = arr.max() if arr.size else 0

    for i in range(n_rows):
        for j in range(n_cols):
            value = int(arr[i, j])

            if max_val > 0 and arr[i, j] > 0.55 * max_val:
                text_color = "white"
            else:
                text_color = "black"

            ax.text(
                j,
                i,
                str(value),
                ha="center",
                va="center",
                fontsize=5,
                color=text_color,
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Cell count", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    fig.tight_layout()
    fig.savefig(out_pdf, dpi=int(PDF_DPI), bbox_inches="tight")
    plt.close(fig)

    del fig, ax, im, arr
    gc.collect()

In [ ]:
# ==============================================================================
# GENERATE ONE PDF PER LEIDEN CLUSTERING COLUMN
# ==============================================================================

generated_files = []

for leiden_key in leiden_keys:
    counts = make_counts(leiden_key)

    out_pdf = PLOT_DIR / (
        f"{safe_filename(leiden_key)}"
        f"_vs_top{top_n}_scimap_populations_counts.pdf"
    )

    plot_counts_heatmap(
        counts=counts,
        leiden_key=leiden_key,
        out_pdf=out_pdf,
    )

    generated_files.append(out_pdf)

    print("Saved:", out_pdf)

print("=" * 80)
print("Generated contingency PDFs:")
for p in generated_files:
    print(" -", p)
print("=" * 80)